In [1]:
import numpy as np
import scipy.io 
import wfdb
import os
import csv

from joblib import Parallel, delayed, parallel_backend
from concurrent.futures import ProcessPoolExecutor
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import time

#os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [2]:
# Importing the custom denoising functions that I wrote
from utils.denoising_functions import Nvar_Calculation_from_std_dev, NLM_1dDarbon, NLM_1dDarbon_2D_full
from utils.denoising_functions import Butterworth_lowpass_filter, remove_baseline_loess
from utils.PlottingFunctions import quick_plot, plot_ecg_stacked
from utils.ExtractingGroups import extract_group, extract_diagnosis_and_group

In [3]:
#Importing the tensorflow package and layers

import tensorflow as tf
from tensorflow.keras.layers import Conv1D, Dense, MaxPooling1D, ReLU, Add, Input
from tensorflow.keras.layers import BatchNormalization, Dropout, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

2026-03-23 19:14:38.944675: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-23 19:14:39.010567: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-23 19:14:40.252135: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
from tensorflow.keras import mixed_precision

# 1. Mixed precision (optional)
policy = mixed_precision.Policy('float32') # Sets the global computation typw to float32 for all layers
mixed_precision.set_global_policy(policy) # Applies the precision policy globally

# 2. Enable memory growth
gpus = tf.config.list_physical_devices('GPU') # Lists all GPUs avaialable to TensorFlow
if gpus:
    for gpu in gpus:
        try:
            # Prevents TensorFlow from allocating all GPU memory upfront.
                # Normally, TensorFlow grabs all GPU memory at the start which can cause issues if there are multiple processes or models
                #With memory growth, TensorFlow starts small and grows memory usage as needed
            tf.config.experimental.set_memory_growth(gpu, True) 
            print(f"Memory growth enabled for {gpu}")
        except RuntimeError as e: # Memory growth can only be set before GPUs are initialized. IF called after TensorFlow already has allocated memory, you will get a Runtime error
            print(e)

# Uncomment if you want to limit GPU memory usage (helps with cuDNN errors)
# tf.config.set_logical_device_configuration(
#     gpus[0],
#     [tf.config.LogicalDeviceConfiguration(memory_limit=20000)]
# )
#tf.config.optimizer.set_jit(False)


Memory growth enabled for PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [5]:
# Original WFDB folder
base_path = "/home/lewisa13/ECG_Data/physionet.org/files/ecg-arrhythmia/1.0.0/WFDBRecords"
output_path = "CleanedECG" # Folder to save denoised signal

os.makedirs(output_path, exist_ok = True)


In [5]:
# --------------------------
# Function to process one file
# --------------------------
def process_ecg_file(mat_path, fs, cutoff, P, PatchHW, save_folder):
    record_id = os.path.splitext(os.path.basename(mat_path))[0]
    output_file = os.path.join(save_folder, f"{record_id}.npy")

    # # Skip if already processed
    # if os.path.exists(output_file):
    #     print("Skipping:", record_id)
    #     return None

    print("Processing:", record_id)

    header_path = mat_path.replace(".mat", ".hea")

    # --------------------------
    # Load ECG
    # --------------------------
    mat_data = scipy.io.loadmat(mat_path)
    ecg = mat_data['val']  # shape (n_leads, n_samples)

    # --------------------------
    # Butterworth low-pass filter
    # --------------------------
    ecg_low_pass = Butterworth_lowpass_filter(ecg, fs, cutoff, order=2)

    # --------------------------
    # LOESS baseline removal (threading-safe)
    # --------------------------
    n_leads = ecg_low_pass.shape[0]

    
    ecg_loess = np.array([
        remove_baseline_loess(ecg_low_pass[ilead, :], fs, frac=0.05, it=3)
        for ilead in range(n_leads)
    ])


    # --------------------------
    # Compute noise variance for NLM
    # --------------------------
    Nvar = Nvar_Calculation_from_std_dev(ecg_loess)

    # --------------------------
    # NLM denoising (non-vectorized per lead)
    # --------------------------
    ecg_denoised = np.array([
        NLM_1dDarbon(ecg_loess[ilead, :], Nvar, P, PatchHW)
        for ilead in range(n_leads)
    ])
    
    # --------------------------
    # Save denoised ECG
    # --------------------------
    #os.makedirs(save_folder, exist_ok=True)
    
    np.save(output_file, ecg_denoised)
    print("Saved:", record_id)
    
    # --------------------------
    # Extract grouedp labels
    # --------------------------
    group_code = extract_group(header_path)
    label_row = [record_id, group_code]

    return label_row

# ---------------------------
# Function to process in batches
#-----------------------------

def process_in_batches(all_mat_files, batch_size, fs, cutoff, P, PatchHW, save_folder):
    total = len(all_mat_files)
    all_label_rows = []

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        #print(f"Processing batch {start}:{end}")

        batch_files = all_mat_files[start:end]

        # for mat_path in batch_files:
        #     process_ecg_file(mat_path, fs, cutoff, P, PatchHW, save_folder)
        #     #breakpoint()
        

        # Parallel processing per batch
        with parallel_backend("loky", n_jobs = 16):
            batch_label_rows = Parallel(verbose = 10)(
                delayed(process_ecg_file)(mat_path, fs, cutoff,P,PatchHW, save_folder)
                    for mat_path in batch_files
            )

        # Remove None values (skipped files)
        batch_label_rows = [row for row in batch_label_rows if row is not None]

        #Add to master list
        all_label_rows.extend(batch_label_rows)
        
    return all_label_rows

# -------------------
# Main function
#--------------------
def main():
    base_path =  "/home/lewisa13/ECG_Data/physionet.org/files/ecg-arrhythmia/1.0.0/WFDBRecords"
    save_folder = "/home/lewisa13/CleanedECG"
    fs = 500          # sampling frequency
    cutoff = 40       # low-pass cutoff
    P = 20            # NLM search window controls how far the algorithm searches for similar patches
    PatchHW = 3       # NLM patch half-width #Patch length = 2* PatchHW +1
    batch_size = 500 # adjust depending on RAM and number of workers

    # Gather all .mat files
    all_mat_files = [
        os.path.join(root,f)
        for root, dirs, files in os.walk(base_path)
        for f in files
        if f.endswith(".mat")
    ]
    print(f"Found {len(all_mat_files)} .mat files")

    start_time = time.time()
    labels = process_in_batches(all_mat_files, batch_size, fs, cutoff, P , PatchHW, save_folder)
    end_time = time.time()

    print(f"\nTotal runtime: {end_time - start_time: .1f}s ({(end_time - start_time)/60:.1f} min)")

    # Save labels to CSV
    df = pd.DataFrame(labels, columns = ["record_id", "group_code"])
    df.to_csv(os.path.join("/home/lewisa13","labels.csv"), index = False)
    print(f"Label CSV saved: {os.path.join(save_folder, 'labels.csv')}")

# --------------------------
# Usage
# --------------------------
if __name__ == "__main__":
    
   main()


Found 45152 .mat files


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   9 tasks      | elapsed:    8.8s


KeyboardInterrupt: 

In [5]:
# --------------------------
# Function to get the diagnosis using the SNOMED code and then label which group it belongs to
# --------------------------
def process_hea_file(hea_path, save_folder): #fs, cutoff, P, PatchHW, 
    record_id = os.path.splitext(os.path.basename(hea_path))[0]
    #output_file = os.path.join(save_folder, f"{record_id}.npy")

    #print("Processing:", record_id)
    
    # --------------------------
    # Extract grouedp labels
    # --------------------------
    snomed, diagnosis, group = extract_diagnosis_and_group(hea_path)

    label_row = [record_id, snomed, diagnosis, group]
    # group_code = extract_group(header_path)
    # label_row = [record_id, group_code]

    return label_row

# ---------------------------
# Function to process in batches
#-----------------------------

def process_in_batches(all_hea_files, batch_size, save_folder):
    total = len(all_hea_files)
    all_label_rows = []

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        #print(f"Processing batch {start}:{end}")

        batch_files = all_hea_files[start:end]

        # for hea_path in batch_files:
        #     process_hea_file(hea_path, save_folder)
        #     #breakpoint()

        # Parallel processing per batch
        with parallel_backend("loky", n_jobs = 16):
            batch_label_rows = Parallel(verbose = 0)(
                delayed(process_hea_file)(hea_path, save_folder) # fs, cutoff,P,PatchHW, save_folder)
                    for hea_path in batch_files
            )
            

        # Remove None values (skipped files)
        batch_label_rows = [row for row in batch_label_rows if row is not None]

        #Add to master list
        all_label_rows.extend(batch_label_rows)
        
    return all_label_rows

# -------------------
# Main function
#--------------------
def main():
    base_path =  "/home/lewisa13/ECG_Data/physionet.org/files/ecg-arrhythmia/1.0.0/WFDBRecords"
    save_folder = "/home/lewisa13"
    # fs = 500          # sampling frequency
    # cutoff = 40       # low-pass cutoff
    # P = 20            # NLM search window
    # PatchHW = 3       # NLM patch half-width
    batch_size = 500 # adjust depending on RAM and number of workers

    # Gather all .mat files
    all_hea_files = [
        os.path.join(root,f)
        for root, dirs, files in os.walk(base_path)
        for f in files
        if f.endswith(".hea")
    ]
    print(f"Found {len(all_hea_files)} .hea files")

    start_time = time.time()
    labels = process_in_batches(all_hea_files,batch_size, save_folder) # batch_size, fs, cutoff, P , PatchHW, save_folder)
    end_time = time.time()

    print(f"\nTotal runtime: {end_time - start_time: .1f}s ({(end_time - start_time)/60:.1f} min)")

    ### Separating the unknown diagnosis
    unknown_labels = [row for row in labels if row[2] == "UNKNOWN"] # SNOMED diagnosis is not one of the 11
    
    
    # Save labels to CSV
    df = pd.DataFrame(labels, columns = ["record_id", "SNOMED","SNOMED diagnosis","group_code"])
    df_filtered = df[df["SNOMED diagnosis"] != "UNKNOWN"]
    df_filtered.to_csv(
        os.path.join("/home/lewisa13","labels_With_No_Unknown_SNOMED_Diagnosis.csv"), index = False)
    print(f"Label CSV saved: {os.path.join('/home/lewisa13','labels_With_No_Unknown_SNOMED_Diagnosis.csv')}")

    # Save unknown labels separately
    if unknown_labels:
        df_unknown = pd.DataFrame(unknown_labels, columns=["record_id", "SNOMED", "SNOMED diagnosis", "group_code"])
        df_unknown.to_csv(os.path.join("/home/lewisa13", "labels_Unknown_SNOMED.csv"), index=False)
        print(f"Unknown labels CSV saved: {os.path.join(save_folder, 'labels_Unknown_SNOMED.csv')}")
    else:
        print("No unknown SNOMED codes found.")

    
# --------------------------
# Usage
# --------------------------
if __name__ == "__main__":
    
   main()


Found 45152 .hea files

Total runtime:  7.7s (0.1 min)
Label CSV saved: /home/lewisa13/labels_With_No_Unknown_SNOMED_Diagnosis.csv
Unknown labels CSV saved: /home/lewisa13/labels_Unknown_SNOMED.csv


In [6]:
# Code to give some statistics of the number of unknown diagnosis (not falling into the 11 groups

df = pd.read_csv("labels_With_No_Unknown_SNOMED_Diagnosis.csv")

# Count rows where column value is "unknown"
count_unknown = (df["SNOMED diagnosis"] == "UNKNOWN").sum()

print(count_unknown)

# # Load your CSV
# df = pd.read_csv("labels_Unknown_SNOMED.csv")

# # Count occurrences of each SNOMED code
# counts = df["SNOMED"].value_counts()

# # Only keep codes that appear more than once
# repeats = counts[counts > 100]

# print(repeats)


0


In [7]:
#-----------------
## Loading the CSV file and adjusting labels
#------------------

df = pd.read_csv("labels_With_No_Unknown_SNOMED_Diagnosis.csv")
df["label"] = df["group_code"]-1

# 1st split: train vs temp (validation + test)
train_df, temp_df = train_test_split(
    df, test_size = 0.2, random_state = 42, stratify = df["label"]
) # stratify ensures that each split has roughly the same proportion of the four groups

# 2nd split: validation vs test (half of the 20% to 10% each)
val_df, test_df = train_test_split(
    temp_df, test_size = 0.5, random_state = 42, stratify = temp_df["label"]
)



In [8]:
#---------------
## Loading the npy ECG signal data
#---------------
signal_folder = "/home/lewisa13/CleanedECG"

def load_npy(record_id, label):
    
    # In TensorFlow pipelines,especially tf.data, record_id can come in different formats
    # bytes (eg. b'12345'), Numpy array (eg array(b'12345', dtype = object) or 0-D arrays
    # Already a Python string or number
    # Convert bytes or NumPy string to Python string
    if isinstance(record_id, bytes):
        record_id_str = record_id.decode() # converts byte string into normal string
    elif isinstance(record_id, np.ndarray):
        # squeeze() removes extra dimensions (eg np.array(['12345']) to np.array('12345')
        # item() extracts the scalar value from a 0-D array, (eg np.array('12345') to '12345')
        record_id_str = record_id.squeeze().item()  # get scalar from 0-d array
    else:
        record_id_str = str(record_id) # Handles python strings, integers, TensorFlow Scalar objects

    # + '.npy' appends the file extension to the string record_id_str
    file_path = os.path.join(signal_folder, record_id_str + ".npy")
    signal = np.load(file_path)  # loads the numpy array of size (12,5000)

    #Transpose
    signal = signal.T # size (5000,12)

    # Performing a center crop. Crops the two ends to get the shape to 4096
    start = (5000 - 4096)//2
    signal = signal[start:start+4096] # (4096,12)

    # normalize
    std = np.std(signal)
    signal = (signal - np.mean(signal)) / (std if std > 1e-6 else 1.0)

    return signal.astype(np.float32),  np.int32(label) #np.array(label, dtype = np.int32)
    

In [9]:
#--------------
## Conversion to tensorflow tensors
#-------------

def tf_wrapper(record_id,label,num_classes):
    # Function is a bridge between NumPy/Pyhton code and a TensorFlow pipeline
    #Lets you use load_npy() function which is a pure Python/Numpy) inside a tf.data workflow
    
    signal,label = tf.numpy_function(
        load_npy,
        [record_id, label], # passes into load_npy
        [tf.float32, tf.int32] # signal expected as tf.float32 and tf.int32
    )

    #signal is now a Tensorflow tensor returned from tf.numpy_function
    # TensorFlow cannot automatically infer the shape of tensors returned by tf.numpy_finction
    signal.set_shape([4096,12]) # Tells TensorFlow that this tensor has shape (4096, 12)
    label.set_shape([]) # Label is a scalar. [] indicates it is a 0-dimensional tensor

    # Convert integer label to one-hot
    # eg if label2 = 2 and num_classes = 4,
    #label = [0, 0, 1, 0]
    label = tf.one_hot(label, depth=num_classes)

    return signal, tf.cast(label, tf.float32) #tf.one_hot may return a different dtype (often float32 but not guarenteed)



def make_dataset(df_split, num_classes, shuffle = False):

    # from_tensor_slices takes one or more arrays of the same length and slices them row-wise into a dataset
        #dataset has each element as a tuple ('1000001,0) ('1000002,2), etc
        #Dataset is of Type <class 'tensorflow.python.data.ops.dataset_ops.TensorSliceDataset'>, so it has multiple attributes
    
    dataset = tf.data.Dataset.from_tensor_slices(
        # takes the pandas Series of record ID's and converts each record ID into a string
        # .values converts the pandas Series to a NumPy array
        (df_split["record_id"].astype(str).values, 
         df_split["label"].values) # Convert the pandas series of labels and converts it into a numpy array
    )

# .shuffle() randomizes the order of elements in the dataset
    # 10000 = buffer size. TensorFlow will keep 10,000 elements in memory and randomly sample from them
    # Larger buffer more uniform shuffling; smaller buffer is faster but less random
    # Shuffling prevents the model from learning spurious order correlations
#. map() applies the tf_wrapper function to each element in the data set
    # lambda r, l: tf_wrapper(r, l, num_classes) is equivalent to writing
        # def temp_function(r, l):
        #     return tf_wrapper(r, l, num_classes) 
        # Takes the two inputs r and and l, calls tf_wrapper with those inputs plus num_classes
        # returns the result
        # .map()expects a function with two arguments function(r,l), lambda acts as a wrapper to "inject num_classes
        # From earlier lines dataset is passes a tuple (record_id_array, label_array), r is the first variable and l is the second variable
# num_parallel_calls = tf.data.AUTOTUNE decides how many threads to use to load and preprocess data in parallel
    # Using autotune speeds up the pipeline so that the GPU isn't waiting for the CPU
# .batch(32) combines 32 individual (signal,label) pairs into one batch
    # After batching, each element of the dataset is now signals.shape(32,4096,12), labels.shape = (32,num_classes)
# .prefetch(tf.data.AUTOTUNE) lets TensorFlow prepare the next batch in the backgroup while GPU is training on the current batch
    # AUTOTUNE automatically selects the best number of batches to prefetch
    
    if shuffle: # shuffle is only used in case of the training set
        dataset = dataset.shuffle(10000)
    
    dataset = (
        dataset
        .map(lambda r, l:tf_wrapper(r,l,num_classes), num_parallel_calls = tf.data.AUTOTUNE)
        .batch(8)
        .prefetch(tf.data.AUTOTUNE)
    )
    return dataset

num_classes = 4
train_dataset = make_dataset(train_df, num_classes, shuffle = True)
val_dataset = make_dataset(val_df, num_classes, shuffle = False)
test_dataset = make_dataset(test_df, num_classes, shuffle = False)

I0000 00:00:1774307718.145732 2483708 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20000 MB memory:  -> device: 0, name: Tesla V100-PCIE-32GB, pci bus id: 0000:3b:00.0, compute capability: 7.0


In [10]:
# Defining the intial block

def initialConvBlock(input):
    # Defines the inital block of the neural network
    #First block has 64 filters, filter length of 16 and stride of 4, "same' is added to determine the required padding
    x1 = Conv1D(filters = 64, kernel_size = 16,padding = 'same',strides = 4)(input)
    x1 = BatchNormalization()(x1)
    x1 = ReLU()(x1)

    return x1

In [11]:
# Defining the resblk block using a function, N_filters is used as an input since the filters double ever two res blocks

def resblk(input, N_filters, stride_size, dropout_rate):
    # resblk has one input which splits into two branches coming from the previous layer

    #Max pooling branch is the first branch
    x1 = MaxPooling1D(pool_size = 4, strides = stride_size, padding = "valid")(input)
    x1 = Conv1D(filters = N_filters, kernel_size = 1, padding = "valid")(x1)

    # Conv layer is the second branch 
    x2 = Conv1D(filters = N_filters, kernel_size = 16, strides = stride_size, padding = "same")(input)
    x2 = BatchNormalization()(x2)
    x2 = ReLU()(x2)
    x2 = Dropout(dropout_rate)(x2)

    x2 = Conv1D(filters = N_filters, kernel_size = 16, strides = 1, padding = "same")(x2)

    # Add the output from the two branches
    x3 = Add()([x1, x2])

    # Apply the batch normalization, ReLU and Dropout layers
    x3 = BatchNormalization()(x3)
    x3 = ReLU()(x3)
    x3 = Dropout(dropout_rate)(x3)
    
    return x3
    

In [12]:
# Putting together the entire network
num_classes = 4

inputs = Input(shape = (4096,12))

convblock1 = initialConvBlock(inputs)

resblk1 = resblk(convblock1, N_filters = 64, stride_size = 4, dropout_rate = 0.5)

resblk2 = resblk(resblk1, N_filters = 64, stride_size = 4, dropout_rate = 0.5)

resblk3 = resblk(resblk2, N_filters = 128, stride_size = 4, dropout_rate = 0.5)

resblk4 = resblk(resblk3, N_filters = 192, stride_size = 4, dropout_rate = 0.5)

flatten_layer = Flatten()(resblk4)

Output_dense_layer = Dense(units = num_classes, activation = 'softmax')(flatten_layer)

model = Model(inputs= inputs, outputs = Output_dense_layer)

model.summary()

# Beta=1.0 makes it a standard F1 score
# average='micro' aggregates TPs, FPs, and FNs across all classes
#micro_f1 = tf.keras.metrics.FBetaScore(beta=1.0, average='micro', threshold=0.5)
weighted_f1 = tf.keras.metrics.FBetaScore(beta=1.0, average='weighted', threshold=0.5)

# precision = tf.keras.Precision()
# recall = tf.keras.Recall()
optimizer = Adam(learning_rate = 0.001)
model.compile(optimizer = optimizer, loss = 'categorical_crossentropy', metrics = ['accuracy', weighted_f1])
# sparse_categorical_crossentropy is used when labels are integers and not one-hot encoded

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 4096, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 1024, 64)  │     12,352 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 1024, 64)  │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 1024, 64)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 256, 64)   │     65,600 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 64)   │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 256, 64)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 256, 64)   │          0 │ re_lu[0][0]       │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256, 64)   │          0 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 256, 64)   │      4,160 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 256, 64)   │     65,600 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256, 64)   │          0 │ conv1d_1[0][0],   │
│                     │                   │            │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 64)   │        256 │ add[0][0]         │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 256, 64)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256, 64)   │          0 │ re_lu_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 64, 64)    │     65,600 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 64, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 64, 64)    │          0 │ dropout_1[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64, 64)    │          0 │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 64, 64)    │      4,160 │ max_pooling1d_1[

 Total params: 1,699,972 (6.48 MB)

 Trainable params: 1,698,052 (6.48 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [13]:

history = model.fit(
    train_dataset,
    validation_data = val_dataset,
    epochs = 20
)

test_loss, test_acc, test_WeightedF1 = model.evaluate(test_dataset)
print(f"Test Accuracy: {test_acc:.4f}, Test Weighted F1 score: {test_WeightedF1:.4f}, Test loss: {test_loss:.4f} " )

Epoch 1/20


2026-03-23 19:15:44.232925: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fc1c40025a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-23 19:15:44.232965: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Tesla V100-PCIE-32GB, Compute Capability 7.0
2026-03-23 19:15:44.443256: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-23 19:15:45.386210: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-03-23 19:15:46.247974: W external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:848] None of the algorithms provided by cuDNN heuristics worked; trying fallback algorithms.
2026-03-23 19:15:46.248006: W external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:851] Conv: %cudnn-conv-bias-activation.47 = (f32[8,64,1,102

UnknownError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/lewisa13/anaconda3/envs/my_env/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/lewisa13/anaconda3/envs/my_env/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/lewisa13/anaconda3/envs/my_env/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/home/lewisa13/anaconda3/envs/my_env/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/home/lewisa13/anaconda3/envs/my_env/lib/python3.12/asyncio/base_events.py", line 618, in run_forever

  File "/home/lewisa13/anaconda3/envs/my_env/lib/python3.12/asyncio/base_events.py", line 1951, in _run_once

  File "/home/lewisa13/anaconda3/envs/my_env/lib/python

In [14]:
for x, y in train_dataset.take(1):
    print(x.shape, y.shape)

print(tf.__version__)
!nvcc --version
!nvidia-smi
    

(8, 4096, 12) (8, 4)
2.20.0
/bin/bash: line 1: nvcc: command not found


2026-03-23 19:16:06.865362: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Mon Mar 23 19:16:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-PCIE-32GB           Off |   00000000:3B:00.0 Off |                    0 |
| N/A   30C    P0             36W /  250W |   20458MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # force CPU
# history = model.fit(train_dataset, validation_data=val_dataset, epochs=1)
# import tensorflow as tf
# with tf.device("/CPU:0"):
#     history = model.fit(train_dataset, validation_data=val_dataset, epochs=1)